In [2]:
import json
from loguru import logger
from datetime import datetime
from typing import Optional, TypedDict, Annotated, Dict, Any, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from pydantic_settings import BaseSettings
from typing import List


class Settings(BaseSettings):
    """应用配置类"""

    APP_NAME: str = "AI小说创作系统"
    APP_VERSION: str = "0.1.0"
    # DEBUG: bool = True
    SECRET_KEY: str = "change_this_in_production"

    OPENAI_API_KEY: str = "omlx-12345678"
    OPENAI_API_BASE: str = "http://localhost:8800/v1"
    OPENAI_MODEL_COMPLEX: str = "Qwen3.5-2B-MLX-4bit"
    OPENAI_MODEL_SIMPLE: str = "Qwen3.5-0.8B-MLX-4bit"

    POSTGRES_HOST: str = "localhost"
    POSTGRES_PORT: int = 5432
    POSTGRES_DB: str = "novel_db"
    POSTGRES_USER: str = "postgres"
    POSTGRES_PASSWORD: str = "postgres_password"

    CHROMA_DB_PATH: str = "./chroma_db"
    CHROMA_COLLECTION_NAME: str = "novel_embeddings"

    OLLAMA_BASE_URL: str = "http://localhost:8800/v1"
    OLLAMA_EMBED_MODEL: str = "all-MiniLM-L6-v2-4bit"

    REDIS_HOST: str = "localhost"
    REDIS_PORT: int = 6379
    REDIS_PASSWORD: str = ""
    REDIS_DB: int = 0

    NEO4J_URI: str = "bolt://localhost:7687"
    NEO4J_USER: str = "neo4j"
    NEO4J_PASSWORD: str = "neo4j_password"

    ALLOWED_ORIGINS: str = "http://localhost:3000,http://127.0.0.1:3000,http://192.168.31.101:3000,http://192.168.31.101:8080"

    LOG_LEVEL: str = "INFO"

    @property
    def database_url(self) -> str:
        """生成PostgreSQL数据库URL"""
        return (
            f"postgresql://{self.POSTGRES_USER}:{self.POSTGRES_PASSWORD}"
            f"@{self.POSTGRES_HOST}:{self.POSTGRES_PORT}/{self.POSTGRES_DB}"
        )

    @property
    def allowed_origins_list(self) -> List[str]:
        """解析CORS允许的源"""
        return [origin.strip() for origin in self.ALLOWED_ORIGINS.split(",")]

    @property
    def redis_url(self) -> str:
        """生成Redis连接URL"""
        if self.REDIS_PASSWORD:
            return f"redis://:{self.REDIS_PASSWORD}@{self.REDIS_HOST}:{self.REDIS_PORT}/{self.REDIS_DB}"
        return f"redis://{self.REDIS_HOST}:{self.REDIS_PORT}/{self.REDIS_DB}"

    class Config:
        """Pydantic配置"""
        env_file = ".env"
        case_sensitive = True


# 创建全局配置实例
settings = Settings()


In [16]:
embed_model = OpenAIEmbeddings(
    model=settings.OLLAMA_EMBED_MODEL,
    base_url=settings.OLLAMA_BASE_URL,
    api_key=settings.OPENAI_API_KEY,
    check_embedding_ctx_length=False,
)

# 测试连接并记录向量维度
test_embedding = embed_model.embed_query("hello")

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field, EmailStr

class AgentType(str, Enum):
    """Agent类型枚举"""
    WORLDVIEW = "worldview"  # 世界观Agent
    CHARACTER = "character"  # 角色Agent
    PLOT = "plot"  # 剧情Agent

class ConsistencyCheckType(str, Enum):
    """一致性检查类型枚举"""
    RULE_ENGINE = "rule_engine"  # 规则引擎
    KNOWLEDGE_GRAPH = "knowledge_graph"  # 知识图谱
    TIMELINE = "timeline"  # 时间线
    EMOTION = "emotion"  # 情绪状态机


class RAGQuery(BaseModel):
    """RAG检索请求"""
    novel_id: int = Field(..., description="小说ID")
    query: str = Field(..., description="检索查询")
    max_chapter: Optional[int] = Field(None, description="最大章节号（用于过滤）")
    top_k: int = Field(3, description="返回Top K结果")

class RAGResult(BaseModel):
    """RAG检索结果"""
    content: str
    metadata: Dict[str, Any]
    score: float

class RAGResponse(BaseModel):
    """RAG检索响应"""
    query: str
    results: List[RAGResult]
    retrieval_method: str  # "hybrid", "vector", "bm25"

class AgentWorkflowStep(BaseModel):
    """单个工作流步骤信息

    用于描述某一步里由哪个Agent/组件执行了什么操作、使用了哪些数据源、产生了什么结果，
    方便前端进行可视化展示（时间线 / DAG / 数据流图）。
    """

    id: str = Field(..., description="步骤ID，需在一次工作流内唯一")
    parent_id: Optional[str] = Field(
        None, description="父步骤ID，用于构建依赖关系/有向图"
    )
    type: str = Field(
        ...,
        description=(
            "步骤类型，例如：agent/rag/graph/llm/consistency/db/index 等，"
            "用于前端选择不同的图标和样式"
        ),
    )
    agent_name: Optional[str] = Field(
        None,
        description="执行该步骤的Agent或组件名称，例如 WorldviewAgent/ConsistencyService",
    )
    title: str = Field(..., description="步骤的简短标题，用于列表或时间线展示")
    description: Optional[str] = Field(
        None, description="对该步骤做了什么的详细说明"
    )

    input: Dict[str, Any] = Field(
        default_factory=dict, description="该步骤的主要输入参数（经过脱敏和裁剪后的概要）"
    )
    output: Dict[str, Any] = Field(
        default_factory=dict, description="该步骤的主要输出结果概要"
    )

    data_sources: Dict[str, Any] = Field(
        default_factory=dict,
        description=(
            "该步骤使用到的数据源，例如：RAG检索片段、知识图谱节点、时间线事件等，"
            "结构上推荐按类别拆分：{'rag_chunks': [...], 'graph_nodes': [...]}"
        ),
    )

    llm: Dict[str, Any] = Field(
        default_factory=dict,
        description=(
            "如果该步骤调用了LLM，这里记录模型名称、温度、prompt模板摘要等信息，"
            "方便前端展示是哪个模型干的这一步。"
        ),
    )

    status: str = Field(
        default="completed",
        description="步骤状态：pending/running/completed/failed/skipped 等",
    )
    started_at: Optional[datetime] = Field(None, description="步骤开始时间（UTC）")
    finished_at: Optional[datetime] = Field(None, description="步骤结束时间（UTC）")
    duration_ms: Optional[int] = Field(
        None, description="执行耗时（毫秒），前端可用于展示性能/瓶颈"
    )

class AgentWorkflowTrace(BaseModel):
    """一次完整工作流运行的追踪信息

    用于描述某次MCP操作/一致性检查/生成调用中，Agent和后端组件的整体协作流程。
    前端可以基于该结构绘制：
    - 时间线视图（按started_at排序）
    - DAG数据流图（根据id/parent_id构建）
    - 性能分析（根据duration_ms聚合）
    """

    run_id: str = Field(..., description="本次工作流运行ID，由后端生成并在整个调用链中传递")
    trigger: str = Field(
        ...,
        description=(
            "触发来源，例如：'mcp.analyze_novel' / 'mcp.optimize_novel' / "
            "'consistency.check_content' 等"
        ),
    )

    novel_id: Optional[int] = Field(None, description="关联的小说ID")
    chapter_id: Optional[int] = Field(None, description="关联的章节ID（如有）")
    user_id: Optional[int] = Field(None, description="触发该工作流的用户ID（如可用）")

    summary: Optional[str] = Field(
        None, description="本次工作流的简要说明，便于前端列表展示"
    )

    steps: List[AgentWorkflowStep] = Field(
        default_factory=list, description="按执行顺序或依赖关系排列的步骤列表"
    )

    created_at: datetime = Field(
        default_factory=datetime.utcnow, description="工作流追踪创建时间"
    )

    class Config:
        """配置选项"""

        # 允许ORM对象转换为该Schema，方便后续如需持久化
        from_attributes = True

class ConsistencyCheckResult(BaseModel):
    """一致性检查结果"""
    check_type: ConsistencyCheckType
    is_valid: bool
    violations: List[str] = Field(default_factory=list)

class AgentOutput(BaseModel):
    """单个Agent的输出"""
    agent_type: AgentType
    content: str
    metadata: Dict[str, Any] = Field(default_factory=dict)

class GenerationRequest(BaseModel):
    """内容生成请求"""
    novel_id: int = Field(..., description="小说ID")
    prompt: str = Field(..., description="剧情提示词")
    chapter: int = Field(..., description="当前章节号")
    current_day: int = Field(1, description="故事当前天数")
    target_length: int = Field(500, description="目标字数")

class GenerationResponse(BaseModel):
    """内容生成响应"""
    novel_id: int
    chapter: int
    final_content: str
    agent_outputs: List[AgentOutput]
    consistency_checks: List[ConsistencyCheckResult]
    retry_count: int = 0
    generated_at: datetime
    worldview_context: List[str] = Field(default_factory=list)
    character_context: List[str] = Field(default_factory=list)
    rag_results: List[Dict[str, Any]] = Field(default_factory=list)
    workflow_trace: Optional[AgentWorkflowTrace] = Field(
        default=None,
        description="本次多Agent生成流程的工作流追踪信息，供前端可视化展示",
    )


## 生成章节

In [17]:
def raw_json(raw):
    start = raw.find("{")
    end = raw.rfind("}") + 1
    json_str = raw[start:end] if start != -1 and end != 0 else raw
    # print(json_str)
    json_str = json_str.replace('\\n', '')
    print(json_str)
    data = json.loads(json_str)

    worldview = str(data.get("worldview") or "").strip()
    main_chars_raw = data.get("main_characters") or data.get("characters") or []
    outline = str(data.get("outline") or "").strip()
    plot_hooks_raw = data.get("plot_hooks") or []

    if isinstance(main_chars_raw, str):
        main_characters = [line.strip() for line in main_chars_raw.split("\n") if line.strip()]
    elif isinstance(main_chars_raw, list):
        main_characters = [str(item).strip() for item in main_chars_raw if str(item).strip()]
    else:
        main_characters = []

    if isinstance(plot_hooks_raw, str):
        plot_hooks = [line.strip() for line in plot_hooks_raw.split("\n") if line.strip()]
    elif isinstance(plot_hooks_raw, list):
        plot_hooks = [str(item).strip() for item in plot_hooks_raw if str(item).strip()]
    else:
        plot_hooks = []

    return worldview, main_characters, outline, plot_hooks
    

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
    (
    "system",
    """你是一名专业的中文网络小说策划编辑，擅长根据简单的想法，生成完整的设定与大纲。

    请你根据用户提供的信息，为这部小说生成：
    - 世界观设定（worldview）：整体世界结构、力量体系、时代背景等，使用多段文字描述；
    - 主要角色列表（main_characters）：3-6个主要角色，每个用一句话概括；
    - 故事大纲（outline）：从开篇到结局的主线设计，可以按段落分点描述；
    - 剧情线索（plot_hooks）：3-8条可以展开的重要伏笔或矛盾。

    输出时请严格使用JSON格式（注意：下面是结构示例，不是内容）：
    {{
    "worldview": "...",
    "main_characters": ["角色1：...", "角色2：..."],
    "outline": "...",
    "plot_hooks": ["线索1", "线索2", "线索3"]
    }}

    不要输出任何解释性文字、注释或前后缀，只输出上述JSON。""",
    ),
    (
    "user",
    """小说标题：{title}
    小说类型：{genre}
    已有简介：{description}
    目标章节数：{target_chapters}
    故事主题/补充说明：{theme}
    """,
    ),
    ]
)

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
    (
    "system",
    """你是一名专业的中文网络小说策划编辑，擅长根据简单的想法，生成完整的设定与大纲。

    请你根据用户提供的信息，为这部小说生成：
    - 世界观设定（worldview）：整体世界结构、力量体系、时代背景等，使用多段文字描述；
    - 主要角色列表（main_characters）：3-6个主要角色，每个用一句话概括；
    - 故事大纲（outline）：从开篇到结局的主线设计，可以按段落分点描述；
    - 剧情线索（plot_hooks）：3-8条可以展开的重要伏笔或矛盾。

    输出时请严格使用JSON格式（注意：下面是结构示例，不是内容）：
    {{
    "worldview": "...",
    "main_characters": ["角色1：...", "角色2：..."],
    "outline": "...",
    "plot_hooks": ["线索1", "线索2", "线索3"]
    }}

    不要输出任何解释性文字、注释或前后缀，只输出上述JSON。""",
    ),
    (
    "user",
    """
    已有简介：{description}
    """,
    ),
    ]
)


basic_model = ChatOpenAI(
    model="Qwen3.5-0.8B-MLX-4bit",
    base_url="http://localhost:8800/v1",
    api_key="omlx-12345678",
    temperature=0.8,
    # max_tokens=3000,
    # timeout=120
)

chain = prompt | basic_model
result = await chain.ainvoke(
    {
        "title": "月影下的承诺",
        "genre": "言情/奇幻",
        "description": "在一个被魔法与爱情交织的世界里，女主角艾莉在一次意外中获得了治愈的能力，而正当她学会控制这项能力时，暗流涌动的危机随之而来。",
        "target_chapters": "1",
        "theme": "探索爱的力量与自我成长，揭示选择之间的孤独与勇气。",
    }
)
result = await chain.ainvoke(
    {
        "description": "李木穿越隋未，机缘巧合下拜张果老为师，一直过着清闲日子。恰逢某位皇后病重，御医束手无策，皇帝为治好皇后，派禁军寻找仙人张果老。无良师父率先跑路，李木被禁军逮个正着，只能硬着头皮进宫为皇后治病，至此开启了他的大唐之旅。  他曾经历隋末乱世的腥风血雨，也曾卷入李密、李渊、李世民等人争夺天下的残酷战争。他看过贞观盛世的万国来朝风华，也见过开元天宝时期的繁荣盛世。  他曾做过禁军，也曾入相拜相。他战过突厥、契丹（可汗诸部），也打过海盗与东南沿海的倭寇（或海上强盗）。  他与房玄龄、杜如晦为朝中挚友，也曾与张旭、李白游走花楼、风流风雅…他是一个被时间长河遗忘的人。"
    }
)

raw = result.content.strip()
raw

'```json\n{\n  "worldview": "李木穿越自隋末，身处乱世，经历了战乱、盛世割据与王朝更迭。他并未身处政治中心，而是在一个由忠奸、权谋与信仰交织的未知世界穿梭。凭借武力和政治智慧，李木不仅完成了从禁军老兵到朝堂重臣的身份转变，更在漫长的岁月里见证了唐帝国的建立。这种跨越数千年时空的流动，既让他拥有极高的格局，也让他保留了穿越者的独特视角，正在构建一个自洽且宏大的历史世界观。",\n  "main_characters": [\n    "李木：被时间遗忘却强行穿越的禁军老兵，拥有政治智慧与武力，致力于成为大唐开国皇帝",\n    "张果老：李木的师傅，传说中拥有超自然力量的仙人，掌控着长生与神通",\n    "张旭：李木的挚友，曾在李密、李渊时期被写诗令作诗\n    "房玄龄：李木与张旭等人的合作者，唐太宗时期著名的宰相",\n    "杜如晦：李木与张旭等人的合作者，谏议大夫，李密、李渊之间的关键政治盟友",\n    "李密、李渊、李世民：李木经历的三个关键历史时期，分别代表了隋、唐、五代。"\n  ],\n  "outline": [\n    "开篇：李木穿越隋末，成为禁军中的被逮者，被迫接受皇后病治的角色，正式踏上大唐之旅。",\n    "发展：在经历隋末乱世与开国伟业后，李木逐渐掌握了政治与军事技术，成为太子、宰相与开国皇帝。",\n    "深化：在后期，李木面临张旭、颜真卿等艺术巨匠的考验，以及后世的挑战，例如唐末的盛世与乱世的交替。",\n    "高潮：李木最终完成皇帝大业，但面对无尽的后世挑战，其命运似乎将永远定格在短暂的盛世与绝响。",\n    "结局：李木成为千古传奇，被时代遗忘，但作为历史见证者，他留下了关于唐帝国的真实记录与评价。"\n  ],\n  "plot_hooks": [\n    "李木成为皇帝后，是否愿意继续与张旭等人进行艺术风格的交流，还是选择留在幕后？",\n    "朝堂上出现的权术斗争中，李木能否找到真正的盟友来对抗未来的反派势力？",\n    "李木去世或遭遇重大变故后，其灵魂如何被托梦给张旭或杜如晦？",\n    "李木与张旭等人终老后，是否会在某个无记无记的朝代间再次穿越？",\n    "李木被后世的某种势力（如鬼谷子等）捕杀，他的结局将如何展开？"\n  ]\n}\n

In [ ]:
worldview, main_characters, outline, plot_hooks = raw_json(raw)

In [ ]:
plot_hooks

In [ ]:
outline

In [ ]:
main_characters

In [ ]:
worldview

## RAG 

In [4]:
import os
import chromadb
from langchain_openai import OpenAIEmbeddings
from chromadb.config import Settings as ChromaSettings
from llama_index.vector_stores.chroma.base import ChromaVectorStore

In [10]:
class RAGService:
    """RAG检索服务类（使用Chroma + Ollama）"""

    def __init__(self):
        """初始化RAG服务"""
        self.available = False
        self.vector_store = None
        self.embed_model: Optional[BaseEmbedding] = None
        self.index = None
        # 当前Embedding向量维度，用于区分不同维度的Chroma集合
        self.embed_dim: Optional[int] = None

        try:
            # 初始化Embedding模型（Ollama本地）
            self._init_embedding()

            # 初始化Chroma向量数据库
            self._init_vector_store()

            self.available = True
            logger.info("✅ RAG服务初始化成功（Chroma + Ollama）")

        except Exception as e:
            logger.warning(f"⚠️ RAG服务初始化失败: {e}")
            logger.warning("RAG功能将不可用，但不影响其他功能")
            self.available = False

    def _init_embedding(self):
        """初始化Embedding模型"""
        try:
            # 尝试使用Ollama本地Embedding
            logger.info("尝试连接Ollama Embedding服务...")
            self.embed_model = OpenAIEmbeddings(
                model_name=settings.OLLAMA_EMBED_MODEL,
                base_url=settings.OLLAMA_BASE_URL,
                api_key=settings.OPENAI_API_KEY,
                request_timeout=30.0,
            )

            # 测试连接并记录向量维度
            test_embedding = self.embed_model.embed_query("测试")
            self.embed_dim = len(test_embedding)
            logger.info(f"✅ Ollama Embedding已连接，向量维度: {self.embed_dim}")

        except Exception as e:
            logger.warning(f"Ollama连接失败: {e}")
            # logger.info("回退到简单Embedding模型...")

            # # 回退方案：使用HuggingFace本地模型（如果Ollama不可用）
            # from llama_index.embeddings.huggingface import HuggingFaceEmbedding

            # self.embed_model = HuggingFaceEmbedding(
            #     model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
            # )

            # # 同样记录维度，保持后续向量库配置一致
            # test_embedding = self.embed_model.get_text_embedding("测试")
            # self.embed_dim = len(test_embedding)
            # logger.info("✅ 使用HuggingFace本地Embedding")
            # logger.info(f"✅ Embedding向量维度: {self.embed_dim}")

    def _init_vector_store(self):
        """初始化Chroma向量数据库"""
        # 确保数据目录存在
        chroma_path = settings.CHROMA_DB_PATH
        os.makedirs(chroma_path, exist_ok=True)

        # 创建Chroma客户端（持久化存储）
        chroma_client = chromadb.PersistentClient(
            path=chroma_path,
            settings=ChromaSettings(
                anonymized_telemetry=False
            )
        )

        # 根据Embedding维度区分集合名称，避免维度不匹配
        base_collection_name = settings.CHROMA_COLLECTION_NAME
        if self.embed_dim:
            collection_name = f"{base_collection_name}_{self.embed_dim}"
        else:
            collection_name = base_collection_name

        chroma_collection = chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={
                "description": (
                    f"小说内容向量存储（维度={self.embed_dim}）" if self.embed_dim else "小说内容向量存储"
                )
            },
        )

        # 创建向量存储
        self.vector_store = ChromaVectorStore(
            chroma_collection=chroma_collection
        )

        logger.info(f"✅ Chroma向量数据库已初始化：{chroma_path}")
        logger.info(f"✅ 集合名称：{collection_name}")

    async def index_content(
        self,
        novel_id: int,
        chapter: int,
        content: str,
        metadata: Optional[Dict[str, Any]] = None
    ) -> bool:
        """
        索引小说内容到向量数据库

        Args:
            novel_id: 小说ID
            chapter: 章节号
            content: 内容文本
            metadata: 额外元数据

        Returns:
            是否成功
        """
        if not self.available:
            logger.warning("RAG服务不可用，跳过索引")
            return False

        try:
            # 分块处理长文本（每500字一块）
            chunks = self._split_text(content, chunk_size=500)

            # 创建文档列表
            documents = []
            for idx, chunk in enumerate(chunks):
                doc_metadata = {
                    "novel_id": novel_id,
                    "chapter": chapter,
                    "chunk_index": idx,
                    **(metadata or {})
                }

                documents.append(
                    Document(
                        text=chunk,
                        metadata=doc_metadata,
                        id_=f"{novel_id}_{chapter}_{idx}"
                    )
                )

            # 创建存储上下文
            storage_context = StorageContext.from_defaults(
                vector_store=self.vector_store
            )

            # 创建或更新索引
            if self.index is None:
                self.index = VectorStoreIndex.from_documents(
                    documents,
                    storage_context=storage_context,
                    embed_model=self.embed_model,
                    show_progress=True
                )
            else:
                # 添加文档到现有索引
                for doc in documents:
                    self.index.insert(doc)

            logger.info(f"✅ 成功索引小说{novel_id}章节{chapter}，共{len(chunks)}个分块")
            return True

        except Exception as e:
            logger.error(f"索引内容失败: {e}")
            return False

    async def hybrid_search(self, query: RAGQuery) -> RAGResponse:
        """
        混合检索（向量检索 + 元数据过滤）

        Args:
            query: 检索请求

        Returns:
            检索响应
        """
        if not self.available or self.index is None:
            logger.warning("RAG服务不可用，返回空结果")
            return RAGResponse(
                query=query.query,
                results=[],
                retrieval_method="hybrid"
            )

        try:
            # 创建检索器
            retriever = self.index.as_retriever(
                similarity_top_k=query.top_k
            )

            # 执行检索
            nodes = retriever.retrieve(query.query)

            # 过滤结果（根据novel_id和max_chapter）
            filtered_nodes = []
            for node in nodes:
                metadata = node.metadata

                # 检查novel_id
                if metadata.get("novel_id") != query.novel_id:
                    continue

                # 检查max_chapter
                if query.max_chapter is not None:
                    if metadata.get("chapter", 0) > query.max_chapter:
                        continue

                filtered_nodes.append(node)

            # 转换为RAGResult
            results = [
                RAGResult(
                    content=node.get_content(),
                    metadata={
                        "chapter": node.metadata.get("chapter"),
                        "chunk_index": node.metadata.get("chunk_index"),
                        **{k: v for k, v in node.metadata.items()
                           if k not in ["chapter", "chunk_index", "novel_id"]}
                    },
                    score=node.score if hasattr(node, 'score') else 0.0
                )
                for node in filtered_nodes[:query.top_k]
            ]

            logger.info(f"✅ 混合检索完成，查询:'{query.query}'，返回{len(results)}条结果")

            return RAGResponse(
                query=query.query,
                results=results,
                retrieval_method="hybrid"
            )

        except Exception as e:
            logger.error(f"混合检索失败: {e}")
            return RAGResponse(
                query=query.query,
                results=[],
                retrieval_method="hybrid"
            )

    async def retrieve_worldview(
        self,
        novel_id: int,
        query: str,
        max_chapter: Optional[int] = None,
    ) -> List[str]:
        """
        检索世界观相关内容

        Args:
            novel_id: 小说ID
            query: 查询内容

        Returns:
            相关内容列表
        """
        rag_query = RAGQuery(
            novel_id=novel_id,
            query=query,
            top_k=3,
            max_chapter=max_chapter,
        )
        response = await self.hybrid_search(rag_query)
        return [result.content for result in response.results]

    async def retrieve_character_info(
        self,
        novel_id: int,
        character_name: str,
        max_chapter: Optional[int] = None,
    ) -> List[str]:
        """
        检索角色相关信息

        Args:
            novel_id: 小说ID
            character_name: 角色名称

        Returns:
            角色相关内容列表
        """
        query = f"{character_name}的性格、外貌、背景"
        rag_query = RAGQuery(
            novel_id=novel_id,
            query=query,
            top_k=3,
            max_chapter=max_chapter,
        )
        response = await self.hybrid_search(rag_query)
        return [result.content for result in response.results]

    def _split_text(self, text: str, chunk_size: int = 500) -> List[str]:
        """
        分割文本为固定大小的块

        Args:
            text: 原始文本
            chunk_size: 块大小（字符数）

        Returns:
            文本块列表
        """
        chunks = []
        for i in range(0, len(text), chunk_size):
            chunk = text[i:i + chunk_size]
            if chunk.strip():
                chunks.append(chunk)
        return chunks

    async def delete_novel_index(self, novel_id: int) -> bool:
        """
        删除小说的所有索引

        Args:
            novel_id: 小说ID

        Returns:
            是否成功
        """
        if not self.available:
            logger.warning("RAG服务不可用，跳过删除")
            return False

        try:
            # Chroma删除需要通过collection的delete方法
            # 注意：这里需要根据metadata过滤删除
            # 由于LlamaIndex的抽象层限制，我们直接操作collection
            if self.vector_store and hasattr(self.vector_store, '_collection'):
                collection = self.vector_store._collection
                # 获取所有该小说的ID
                results = collection.get(
                    where={"novel_id": novel_id}
                )
                if results and results.get('ids'):
                    collection.delete(ids=results['ids'])
                    logger.info(f"✅ 删除小说{novel_id}的所有索引")
                    return True

            logger.warning(f"未找到小说{novel_id}的索引")
            return False

        except Exception as e:
            logger.error(f"删除索引失败: {e}")
            return False

    async def cleanup_novel_vectors(self, novel_id: int) -> int:
        """
        清理小说的向量数据
        
        Args:
            novel_id: 小说ID
            
        Returns:
            清理的向量数量
        """
        if not self.available:
            logger.warning("RAG服务不可用，跳过清理向量")
            return 0

        try:
            if self.vector_store and hasattr(self.vector_store, '_collection'):
                collection = self.vector_store._collection
                # 获取所有该小说的向量
                results = collection.get(
                    where={"novel_id": novel_id}
                )
                
                count = 0
                if results and results.get('ids'):
                    count = len(results['ids'])
                    collection.delete(ids=results['ids'])
                    logger.info(f"✅ 清理小说{novel_id}的{count}个向量")
                
                return count
            
            return 0
            
        except Exception as e:
            logger.error(f"清理向量数据失败: {e}")
            return 0

    async def cleanup_novel_graph(self, novel_id: int) -> int:
        """
        清理小说的知识图谱数据
        
        Args:
            novel_id: 小说ID
            
        Returns:
            清理的图谱节点数量
        """
        # 这里可以添加Neo4j或其他图数据库的清理逻辑
        # 目前返回0，表示没有图谱数据需要清理
        try:
            # TODO: 实现知识图谱清理
            # 例如：删除Neo4j中相关的节点和关系
            logger.info(f"知识图谱清理功能待实现，小说ID: {novel_id}")
            return 0
        except Exception as e:
            logger.error(f"清理知识图谱失败: {e}")
            return 0

    async def cleanup_novel_cache(self, novel_id: int) -> int:
        """
        清理小说的缓存数据
        
        Args:
            novel_id: 小说ID
            
        Returns:
            清理的缓存条目数量
        """
        try:
            # 这里可以清理Redis缓存或其他缓存系统
            # 目前只是模拟清理
            cache_keys = [
                f"novel:{novel_id}:worldview",
                f"novel:{novel_id}:characters",
                f"novel:{novel_id}:plot",
                f"novel:{novel_id}:style",
            ]
            
            # TODO: 实现实际的缓存清理
            # 例如：redis_client.delete(*cache_keys)
            logger.info(f"缓存清理功能待实现，小说ID: {novel_id}")
            return len(cache_keys)
            
        except Exception as e:
            logger.error(f"清理缓存失败: {e}")
            return 0

    async def cleanup_chapter_data(self, novel_id: int, chapter_id: int) -> bool:
        """
        清理特定章节的数据
        
        Args:
            novel_id: 小说ID
            chapter_id: 章节ID
            
        Returns:
            是否成功
        """
        if not self.available:
            logger.warning("RAG服务不可用，跳过清理章节数据")
            return False

        try:
            if self.vector_store and hasattr(self.vector_store, '_collection'):
                collection = self.vector_store._collection
                # 删除特定章节的向量数据
                results = collection.get(
                    where={
                        "novel_id": novel_id,
                        "chapter": chapter_id
                    }
                )
                
                if results and results.get('ids'):
                    collection.delete(ids=results['ids'])
                    logger.info(f"✅ 清理小说{novel_id}章节{chapter_id}的向量数据")
                
                return True
            
            return False
            
        except Exception as e:
            logger.error(f"清理章节数据失败: {e}")
            return False


# 创建全局实例
rag_service = RAGService()

2026-04-03 23:00:18.480 | INFO     | __main__:_init_embedding:32 - 尝试连接Ollama Embedding服务...
2026-04-03 23:00:18.524 | WARNING  | __main__:_init_embedding:46 - Ollama连接失败: Embeddings.create() got an unexpected keyword argument 'model_name'
2026-04-03 23:00:18.529 | INFO     | __main__:_init_vector_store:97 - ✅ Chroma向量数据库已初始化：./chroma_db
2026-04-03 23:00:18.529 | INFO     | __main__:_init_vector_store:98 - ✅ 集合名称：novel_embeddings
2026-04-03 23:00:18.530 | INFO     | __main__:__init__:21 - ✅ RAG服务初始化成功（Chroma + Ollama）


## 一致性检查

In [ ]:
"""一致性检查服务

实现四层防护机制：规则引擎、知识图谱、时间线管理、情绪状态机，
并提供 Agent 工作流追踪信息，便于前端可视化展示检查流程和数据流。
"""
import re
from typing import Dict, Any, List, Optional
from datetime import datetime
from neo4j import GraphDatabase
from loguru import logger

# ========== 规则引擎 ==========
class RuleEngine:
    """规则引擎：验证硬规则"""

    def __init__(self):
        self.rules = {}

    def add_rule(self, name: str, value: Any):
        """添加规则"""
        self.rules[name] = value

    def validate(self, content: str) -> Dict[str, Any]:
        """
        验证内容是否违反硬规则

        Args:
            content: 待验证内容

        Returns:
            验证结果
        """
        violations = []

        # 检查魔法等级
        if "魔法等级上限" in self.rules:
            matches = re.findall(r"(\d+)级魔法师", content)
            for match in matches:
                level = int(match)
                if level > self.rules["魔法等级上限"]:
                    violations.append(
                        f"魔法等级{level}超出上限{self.rules['魔法等级上限']}"
                    )

        # 检查飞行速度
        if "飞行速度上限" in self.rules:
            matches = re.findall(r"以(\d+)(?:公里|千米)(?:每|\/)?小时", content)
            for match in matches:
                speed = int(match)
                if speed > self.rules["飞行速度上限"]:
                    violations.append(
                        f"飞行速度{speed}km/h超出上限{self.rules['飞行速度上限']}km/h"
                    )

        return {
            "is_valid": len(violations) == 0,
            "violations": violations
        }


# ========== 知识图谱 ==========
class KnowledgeGraph:
    """知识图谱：验证角色关系和地理位置

    当前主要支持：
    - 解析文本里的简单角色关系句式（如“李青山是苏言的老师”“李青山与苏言是朋友”）
    - 将关系归一化为 friend/ally/enemy/mentor/lover 等语义并写入 Neo4j
    - 检测与已有关系的冲突（例如 friend vs enemy）
    """

    # 关系同义词归一化表
    RELATION_SYNONYMS = {
        "朋友": "friend",
        "好友": "friend",
        "挚友": "friend",
        "盟友": "ally",
        "同盟": "ally",
        "敌人": "enemy",
        "仇人": "enemy",
        "宿敌": "enemy",
        "老师": "mentor",
        "师父": "mentor",
        "导师": "mentor",
        "徒弟": "disciple",
        "学生": "disciple",
        "恋人": "lover",
        "爱人": "lover",
        "伴侣": "lover",
    }

    # 关系的反向映射
    RELATION_INVERSE = {
        "mentor": "disciple",
        "disciple": "mentor",
        "lover": "lover",
        "friend": "friend",
        "ally": "ally",
        "enemy": "enemy",
    }

    # 被视为互斥的关系集合
    RELATION_CONFLICTS = [
        {"friend", "enemy"},
        {"ally", "enemy"},
        {"lover", "enemy"},
    ]

    def __init__(self):
        try:
            self.driver = GraphDatabase.driver(
                settings.NEO4J_URI,
                auth=(settings.NEO4J_USER, settings.NEO4J_PASSWORD)
            )
            logger.info("成功连接Neo4j")
        except Exception as e:
            logger.warning(f"Neo4j连接失败: {e}，将跳过知识图谱检查")
            self.driver = None

    def _normalize_relation(self, relation: Optional[str]) -> Optional[str]:
        """将自然语言关系归一化为内部标识，如“朋友”->"friend""" 
        if not relation:
            return None
        return self.RELATION_SYNONYMS.get(relation.strip())

    def _is_conflict(self, relation_a: str, relation_b: str) -> bool:
        """判断两个关系是否属于互斥关系"""
        if relation_a == relation_b:
            return False
        for conflict in self.RELATION_CONFLICTS:
            if relation_a in conflict and relation_b in conflict:
                return True
        return False

    def _extract_relationships(self, content: str) -> List[Dict[str, str]]:
        """从文本中抽取简单角色关系

        支持的句式示例：
        - "李青山是苏言的老师"
        - "李青山与苏言是朋友"
        """
        if not content:
            return []

        relationships: List[Dict[str, str]] = []

        # 句式1：A是B的X
        pattern_possessive = re.compile(
            r"(?P<a>[\u4e00-\u9fa5A-Za-z]{2,})是(?P<b>[\u4e00-\u9fa5A-Za-z]{2,})的(?P<relation>[\u4e00-\u9fa5A-Za-z]{1,4})"
        )

        # 句式2：A与B是X
        pattern_pair = re.compile(
            r"(?P<a>[\u4e00-\u9fa5A-Za-z]{2,})与(?P<b>[\u4e00-\u9fa5A-Za-z]{2,})是(?P<relation>[\u4e00-\u9fa5A-Za-z]{1,4})"
        )

        for match in pattern_possessive.finditer(content):
            relation = self._normalize_relation(match.group("relation"))
            if not relation:
                continue
            relationships.append(
                {
                    "source": match.group("a"),
                    "target": match.group("b"),
                    "relation": relation,
                }
            )

        for match in pattern_pair.finditer(content):
            relation = self._normalize_relation(match.group("relation"))
            if not relation:
                continue
            relationships.append(
                {
                    "source": match.group("a"),
                    "target": match.group("b"),
                    "relation": relation,
                }
            )

        return relationships

    def _upsert_relationship(self, novel_id: int, char_a: str, char_b: str, relation: str) -> None:
        """向 Neo4j 中写入或更新角色关系（带 novel_id 作用域）"""
        if not self.driver:
            return

        with self.driver.session() as session:
            session.run(
                "MERGE (a:Character {name: $name_a, novel_id: $novel_id}) "
                "MERGE (b:Character {name: $name_b, novel_id: $novel_id}) "
                "MERGE (a)-[r:RELATION {novel_id: $novel_id}]->(b) "
                "SET r.type = $relation",
                name_a=char_a,
                name_b=char_b,
                relation=relation,
                novel_id=novel_id,
            )

    def add_relationship(self, novel_id: int, char_a: str, char_b: str, relation: str) -> None:
        """添加角色关系（会自动写入正向与反向关系）"""
        normalized = self._normalize_relation(relation)
        if not normalized or not self.driver:
            return

        self._upsert_relationship(novel_id, char_a, char_b, normalized)

        inverse = self.RELATION_INVERSE.get(normalized)
        if inverse:
            self._upsert_relationship(novel_id, char_b, char_a, inverse)

    def validate_relationship(self, novel_id: int, char_a: str, char_b: str, new_relation: str) -> Dict[str, Any]:
        """验证新关系是否与已有关系冲突"""
        normalized = self._normalize_relation(new_relation)
        if not normalized or not self.driver:
            return {"is_valid": True}

        with self.driver.session() as session:
            result = session.run(
                "MATCH (a:Character {name: $name_a, novel_id: $novel_id})-"
                "[r:RELATION {novel_id: $novel_id}]->(b:Character {name: $name_b, novel_id: $novel_id}) "
                "RETURN r.type AS relation",
                name_a=char_a,
                name_b=char_b,
                novel_id=novel_id,
            )
            existing_relations = [record["relation"] for record in result]

        for existing in existing_relations:
            if self._is_conflict(existing, normalized):
                return {
                    "is_valid": False,
                    "reason": f"{char_a}和{char_b}已有关系{existing_relations}，与新关系'{new_relation}'矛盾",
                }

        return {"is_valid": True, "normalized": normalized}

    def analyze_content(self, novel_id: int, content: str) -> Dict[str, Any]:
        """从内容中抽取角色关系并写入图谱，同时返回冲突信息"""
        if not self.driver:
            return {"violations": [], "extracted": []}

        relationships = self._extract_relationships(content)
        if not relationships:
            return {"violations": [], "extracted": []}

        violations: List[str] = []
        extracted: List[Dict[str, str]] = []

        for rel in relationships:
            result = self.validate_relationship(
                novel_id=novel_id,
                char_a=rel["source"],
                char_b=rel["target"],
                new_relation=rel["relation"],
            )
            if not result.get("is_valid", True):
                violations.append(result.get("reason") or "角色关系与既有设定冲突")
                continue

            normalized = result.get("normalized") or rel["relation"]
            self.add_relationship(novel_id, rel["source"], rel["target"], normalized)
            extracted.append({**rel, "relation": normalized})

        return {"violations": violations, "extracted": extracted}

    def close(self):
        """关闭连接"""
        if self.driver:
            self.driver.close()


# ========== 时间线管理器 ==========
class TimelineManager:
    """时间线管理器：验证时间一致性"""

    def __init__(self):
        # 存储每个小说的事件时间线
        self.timelines: Dict[int, List[tuple]] = {}

    def add_event(self, novel_id: int, day: int, event: str):
        """添加事件到时间线"""
        if novel_id not in self.timelines:
            self.timelines[novel_id] = []

        self.timelines[novel_id].append((day, event))
        self.timelines[novel_id].sort(key=lambda x: x[0])

    def validate_new_event(
        self,
        novel_id: int,
        day: int,
        event: str
    ) -> Dict[str, Any]:
        """
        验证新事件是否违反时间线

        Args:
            novel_id: 小说ID
            day: 事件发生天数
            event: 事件描述

        Returns:
            验证结果
        """
        if novel_id not in self.timelines or not self.timelines[novel_id]:
            return {"is_valid": True}

        last_day, last_event = self.timelines[novel_id][-1]

        # 检查1：新事件不能早于最后事件
        if day < last_day:
            return {
                "is_valid": False,
                "reason": f"时间倒退：新事件在第{day}天，但最新事件在第{last_day}天"
            }

        # 检查2：地理位置移动是否合理（简化检查）
        # 注意：同一天内可以在不同地点发生事件（如从城市出发到郊外），只要时间间隔足够即可
        cities = re.findall(r"([A-Za-z\u4e00-\u9fa5]{2,}(?:城|镇|村|国))", event)
        last_cities = re.findall(r"([A-Za-z\u4e00-\u9fa5]{2,}(?:城|镇|村|国))", last_event)

        if cities and last_cities:
            # 如果位置改变，检查时间间隔是否足够
            # 一般来说，同一天内的位置改变是合理的（如从城市到郊外）
            # 只有当位置改变且时间间隔为0时才需要检查是否合理
            if cities[0] != last_cities[0] and day == last_day:
                # 同一天内位置改变，需要检查是否在同一个事件中描述
                # 这种情况下，应该允许（因为可能是同一个事件中的多个地点）
                # 所以这里不做限制
                pass

        return {"is_valid": True}

    def get_timeline(self, novel_id: int) -> List[tuple]:
        """获取小说的时间线"""
        return self.timelines.get(novel_id, [])


# ========== 情绪状态机 ==========
class EmotionStateMachine:
    """情绪状态机：验证角色情绪转换"""

    def __init__(self):
        # 定义允许的情绪转换
        self.transitions = {
            "平静": ["高兴", "悲伤", "愤怒", "惊讶"],
            "高兴": ["平静", "兴奋", "惊讶"],
            "悲伤": ["平静", "绝望", "愤怒"],
            "愤怒": ["平静", "暴怒", "悲伤"],
            "暴怒": ["愤怒", "悲伤"],  # 不能直接平静
            "兴奋": ["高兴", "平静"],
            "绝望": ["悲伤", "平静"],
            "惊讶": ["平静", "高兴", "恐惧"],
            "恐惧": ["平静", "惊讶", "绝望"]
        }

        # 存储每个角色的当前情绪
        self.current_emotions: Dict[str, str] = {}

    def set_emotion(self, character: str, emotion: str):
        """设置角色情绪"""
        self.current_emotions[character] = emotion

    def validate_transition(
        self,
        character: str,
        new_emotion: str
    ) -> Dict[str, Any]:
        """
        验证情绪转换是否合理

        Args:
            character: 角色名
            new_emotion: 新情绪

        Returns:
            验证结果
        """
        current = self.current_emotions.get(character, "平静")

        if new_emotion in self.transitions.get(current, []):
            self.current_emotions[character] = new_emotion
            return {"is_valid": True}
        else:
            return {
                "is_valid": False,
                "reason": f"{character}的情绪不能从'{current}'直接转换到'{new_emotion}'"
            }


# ========== 一致性检查服务 ==========
class ConsistencyService:
    """一致性检查服务：整合四层防护机制"""

    def __init__(self):
        self.rule_engine = RuleEngine()
        self.knowledge_graph = KnowledgeGraph()
        self.timeline_manager = TimelineManager()
        self.emotion_machine = EmotionStateMachine()

    def init_worldview_rules(self, novel_id: int, rules: Dict[str, Any]):
        """初始化小说的世界观规则"""
        for name, value in rules.items():
            self.rule_engine.add_rule(name, value)
        logger.info(f"小说{novel_id}初始化了{len(rules)}条世界观规则")

    async def check_content(
        self,
        novel_id: int,
        content: str,
        chapter: int,
        current_day: int
    ) -> Dict[str, Any]:
        """
        执行完整的一致性检查

        Args:
            novel_id: 小说ID
            content: 待检查内容
            chapter: 章节号
            current_day: 当前天数

        Returns:
            检查结果
        """
        violations: List[str] = []
        checks_performed: List[str] = []
        steps: List[AgentWorkflowStep] = []

        # 生成本次检查的运行ID
        run_id = f"consistency-{novel_id}-{chapter}-{int(datetime.utcnow().timestamp() * 1000)}"

        # 第1层：规则引擎检查
        rule_start = datetime.utcnow()
        rule_result = self.rule_engine.validate(content)
        rule_end = datetime.utcnow()
        checks_performed.append("rule_engine")
        if not rule_result["is_valid"]:
            violations.extend(rule_result["violations"])
            logger.warning(f"规则引擎检测到{len(rule_result['violations'])}个违规")

        steps.append(
            AgentWorkflowStep(
                id="rule_engine",
                parent_id=None,
                type="rule_engine",
                agent_name="RuleEngine",
                title="规则引擎检查",
                description="根据预设的世界观硬规则检查文本内容是否违规",
                input={
                    "novel_id": novel_id,
                    "chapter": chapter,
                    "current_day": current_day,
                },
                output={
                    "is_valid": rule_result["is_valid"],
                    "violation_count": len(rule_result["violations"]),
                },
                data_sources={},
                llm={},
                status="completed",
                started_at=rule_start,
                finished_at=rule_end,
                duration_ms=int((rule_end - rule_start).total_seconds() * 1000),
            )
        )

        # 第2层：知识图谱检查（角色关系）
        kg_start = datetime.utcnow()
        kg_result = self.knowledge_graph.analyze_content(novel_id, content)
        kg_end = datetime.utcnow()
        checks_performed.append("knowledge_graph")
        if kg_result.get("violations"):
            violations.extend(kg_result["violations"])
            logger.warning(f"知识图谱检测到{len(kg_result['violations'])}个角色关系冲突")

        steps.append(
            AgentWorkflowStep(
                id="knowledge_graph",
                parent_id="rule_engine",
                type="graph",
                agent_name="KnowledgeGraph",
                title="知识图谱检查角色关系",
                description="从文本中抽取角色关系，写入Neo4j并检测与既有关系的冲突",
                input={
                    "novel_id": novel_id,
                    "chapter": chapter,
                },
                output={
                    "violation_count": len(kg_result.get("violations", [])),
                    "extracted_count": len(kg_result.get("extracted", [])),
                },
                data_sources={
                    "extracted_relationships": kg_result.get("extracted", [])[:10]
                },
                llm={},
                status="completed",
                started_at=kg_start,
                finished_at=kg_end,
                duration_ms=int((kg_end - kg_start).total_seconds() * 1000),
            )
        )

        # 第3层：时间线检查
        timeline_start = datetime.utcnow()
        timeline_result = self.timeline_manager.validate_new_event(
            novel_id, current_day, content
        )
        timeline_end = datetime.utcnow()
        checks_performed.append("timeline")
        if not timeline_result["is_valid"]:
            violations.append(timeline_result["reason"])
            logger.warning(f"时间线检测到违规：{timeline_result['reason']}")
        else:
            # 验证通过，添加到时间线
            self.timeline_manager.add_event(novel_id, current_day, content)

        steps.append(
            AgentWorkflowStep(
                id="timeline",
                parent_id="knowledge_graph",
                type="timeline",
                agent_name="TimelineManager",
                title="时间线检查",
                description="验证事件时间顺序和地理移动是否合理",
                input={
                    "novel_id": novel_id,
                    "current_day": current_day,
                },
                output={
                    "is_valid": timeline_result["is_valid"],
                    "reason": timeline_result.get("reason"),
                },
                data_sources={},
                llm={},
                status="completed",
                started_at=timeline_start,
                finished_at=timeline_end,
                duration_ms=int((timeline_end - timeline_start).total_seconds() * 1000),
            )
        )

        # 第4层：情绪状态机检查（暂未实现，保留为空步骤占位便于前端展示流程完整性）
        checks_performed.append("emotion_state")

        steps.append(
            AgentWorkflowStep(
                id="emotion_state",
                parent_id="timeline",
                type="emotion_state",
                agent_name="EmotionStateMachine",
                title="情绪状态机检查（占位）",
                description="预留用于未来的情绪状态机一致性检查，目前暂未实现",
                input={},
                output={},
                data_sources={},
                llm={},
                status="skipped",
                started_at=None,
                finished_at=None,
                duration_ms=None,
            )
        )

        # 构建工作流追踪
        workflow_trace = AgentWorkflowTrace(
            run_id=run_id,
            trigger="consistency.check_content",
            novel_id=novel_id,
            chapter_id=chapter,
            user_id=None,
            summary=f"小说{novel_id} 第{chapter}章的一致性检查",
            steps=steps,
        )

        return {
            "has_conflict": len(violations) > 0,
            "violations": violations,
            "checks_performed": checks_performed,
            "knowledge_graph_extracted": kg_result.get("extracted", []),
            # 分层结果，供流式接口和前端可视化使用
            "layer_results": {
                "rule_engine": rule_result,
                "knowledge_graph": kg_result,
                "timeline": timeline_result,
            },
            # 完整工作流追踪信息
            "workflow_trace": workflow_trace.model_dump(),
        }

    async def check_content_stream(
        self,
        novel_id: int,
        content: str,
        chapter: int,
        current_day: int,
    ):
        """以流式形式执行一致性检查，按步骤产出事件。

        该方法主要用于 SSE 接口，整体检查逻辑与 ``check_content`` 保持一致，
        但会在每一层检查完成后产出一个事件，最后再产出汇总结果。

        Yields:
            dict: 形如 {"type": "layer" | "summary", ...} 的事件字典。
        """

        # 复用非流式检查逻辑，确保结果一致，并在此基础上拆分流式事件
        result = await self.check_content(
            novel_id=novel_id,
            content=content,
            chapter=chapter,
            current_day=current_day,
        )

        layer_results = result.get("layer_results", {})

        # 第1层：规则引擎
        rule_result = layer_results.get("rule_engine", {})
        yield {
            "type": "layer",
            "layer": "rule_engine",
            "status": "ok" if rule_result.get("is_valid", True) else "violation",
            "violations": rule_result.get("violations", []),
        }

        # 第2层：知识图谱
        kg_result = layer_results.get("knowledge_graph", {})
        kg_violations = kg_result.get("violations", [])
        yield {
            "type": "layer",
            "layer": "knowledge_graph",
            "status": "ok" if not kg_violations else "violation",
            "violations": kg_violations,
            "extracted": kg_result.get("extracted", []),
        }

        # 第3层：时间线
        timeline_result = layer_results.get("timeline", {})
        timeline_is_valid = timeline_result.get("is_valid", True)
        yield {
            "type": "layer",
            "layer": "timeline",
            "status": "ok" if timeline_is_valid else "violation",
            "violations": ([] if timeline_is_valid else [timeline_result.get("reason")]),
        }

        # 第4层：情绪状态机（占位）
        yield {
            "type": "layer",
            "layer": "emotion_state",
            "status": "skipped",
            "violations": [],
        }

        # 最终汇总事件，附带workflow_trace，便于前端获取完整工作流
        summary = {
            "type": "summary",
            "has_conflict": result.get("has_conflict", False),
            "violations": result.get("violations", []),
            "checks_performed": result.get("checks_performed", []),
            "knowledge_graph_extracted": result.get("knowledge_graph_extracted", []),
            "workflow_trace": result.get("workflow_trace"),
        }
        yield summary


# 创建全局实例
consistency_service = ConsistencyService()


## 多Agent协作 章节内容

In [29]:
class NovelGenerationState(TypedDict):
    """小说生成工作流状态"""
    # 输入
    novel_id: int
    prompt: str
    chapter: int
    current_day: int
    target_length: int

    # Agent输出
    worldview_output: str
    character_output: str
    plot_output: str

    # 检索到的上下文
    worldview_context: List[str]
    character_context: List[str]

    # 一致性检查结果
    consistency_result: Dict[str, Any]

    # 重试次数
    retry_count: int

    # 工作流步骤（序列化后的AgentWorkflowStep字典列表）
    workflow_steps: List[Dict[str, Any]]

class AgentService:
    """多Agent服务类"""

    def __init__(self):
        """初始化Agent服务"""
        # 初始化LLM
        self.llm_complex = ChatOpenAI(
            model=settings.OPENAI_MODEL_COMPLEX,
            api_key=settings.OPENAI_API_KEY,
            base_url=settings.OPENAI_API_BASE,
            temperature=0.8
        )
        self.llm_simple = ChatOpenAI(
            model=settings.OPENAI_MODEL_SIMPLE,
            api_key=settings.OPENAI_API_KEY,
            base_url=settings.OPENAI_API_BASE,
            temperature=0.7
        )

        # 构建工作流图
        self.workflow = self._build_workflow()

    def _build_workflow(self) -> StateGraph:
        """构建LangGraph工作流"""
        # 创建状态图
        workflow = StateGraph(NovelGenerationState)

        # 添加节点
        workflow.add_node("retrieve_context", self._retrieve_context)
        workflow.add_node("agent_a_worldview", self._agent_a_worldview)
        workflow.add_node("agent_b_character", self._agent_b_character)
        workflow.add_node("agent_c_plot", self._agent_c_plot)
        workflow.add_node("consistency_check", self._consistency_check)

        # 定义执行顺序
        workflow.set_entry_point("retrieve_context")
        workflow.add_edge("retrieve_context", "agent_a_worldview")
        workflow.add_edge("agent_a_worldview", "agent_b_character")
        workflow.add_edge("agent_b_character", "agent_c_plot")
        workflow.add_edge("agent_c_plot", "consistency_check")

        # 条件分支：一致性检查失败则重试
        workflow.add_conditional_edges(
            "consistency_check",
            self._should_retry,
            {
                "retry": "agent_c_plot",  # 回退到Agent C重新生成
                "end": END
            }
        )

        return workflow.compile()

    async def _retrieve_context(self, state: NovelGenerationState) -> Dict:
        """
        检索上下文节点
        从RAG中检索世界观和角色信息
        """
        logger.info(f"检索上下文：小说{state['novel_id']}，提示词:'{state['prompt']}'")

        # 为避免剧透，RAG只检索当前章节及之前的内容
        current_chapter = state.get("chapter", 1)
        max_chapter = current_chapter if current_chapter > 0 else None

        # 检索世界观相关内容
        step_start = datetime.utcnow()
        worldview_context = await rag_service.retrieve_worldview(
            novel_id=state["novel_id"],
            query=state["prompt"],
            max_chapter=max_chapter,
        )

        # 检索角色相关内容（假设提示词中包含角色名）
        # TODO: 实际应该通过NER提取角色名
        character_context = await rag_service.retrieve_character_info(
            novel_id=state["novel_id"],
            character_name="主角",  # 简化处理
            max_chapter=max_chapter,
        )
        step_end = datetime.utcnow()

        # 记录工作流步骤
        steps = state.get("workflow_steps", [])
        retrieve_step = AgentWorkflowStep(
            id="retrieve_context",
            parent_id=None,
            type="rag",
            agent_name="RAGService",
            title="检索上下文",
            description="从RAG中检索世界观和角色相关上下文，避免剧透。",
            input={
                "novel_id": state["novel_id"],
                "chapter": state["chapter"],
                "prompt": state["prompt"],
                "max_chapter": max_chapter,
            },
            output={
                "worldview_chunks": len(worldview_context or []),
                "character_chunks": len(character_context or []),
            },
            data_sources={
                "worldview_context": worldview_context[:5],
                "character_context": character_context[:5],
            },
            llm={},
            status="completed",
            started_at=step_start,
            finished_at=step_end,
            duration_ms=int((step_end - step_start).total_seconds() * 1000),
        )
        steps.append(retrieve_step.model_dump())

        return {
            "worldview_context": worldview_context,
            "character_context": character_context,
            "workflow_steps": steps,
        }

    async def _agent_a_worldview(self, state: NovelGenerationState) -> Dict:
        """
        Agent A：世界观描写
        专注于环境渲染、氛围营造、魔法体系描写
        """
        logger.info("Agent A（世界观）开始工作")

        # 构建提示词
        prompt = ChatPromptTemplate.from_messages([
            ("system", """你是一位专业的小说世界观描写专家。你的任务是根据剧情提示，描写场景的环境、氛围和相关的世界观元素（如魔法、科技等）。

            要求：
            1. 专注于环境和氛围的渲染
            2. 融入世界观设定（魔法体系、地理环境等）
            3. 控制在150-200字
            4. 不要涉及角色对话和具体剧情
            5. 直接描写环境，严禁使用"这里是..."、"场景展示了..."等说明性语言，沉浸式描写

            世界观上下文：
            {worldview_context}
            """),
                        ("user", "剧情提示：{prompt}\n\n请描写场景的世界观和环境氛围。")
                    ])

        # 调用LLM
        step_start = datetime.utcnow()
        chain = prompt | self.llm_simple
        response = await chain.ainvoke({
            "prompt": state["prompt"],
            "worldview_context": "\n".join(state.get("worldview_context", ["无相关世界观信息"]))
        })
        step_end = datetime.utcnow()

        worldview_output = response.content
        logger.info(f"Agent A输出：{worldview_output[:50]}...")

        steps = state.get("workflow_steps", [])
        step = AgentWorkflowStep(
            id="agent_a_worldview",
            parent_id="retrieve_context",
            type="llm",
            agent_name="AgentAWorldview",
            title="世界观描写Agent",
            description="基于检索到的世界观上下文生成环境与氛围描写。",
            input={
                "prompt": state["prompt"],
                "target_length_hint": "150-200",
            },
            output={
                "preview": worldview_output[:80],
                "length": len(worldview_output),
            },
            data_sources={
                "worldview_context": state.get("worldview_context", [])[:5],
            },
            llm={
                "model": settings.OPENAI_MODEL_SIMPLE,
                "temperature": 0.7,
            },
            status="completed",
            started_at=step_start,
            finished_at=step_end,
            duration_ms=int((step_end - step_start).total_seconds() * 1000),
        )
        steps.append(step.model_dump())

        return {"worldview_output": worldview_output, "workflow_steps": steps}

    async def _agent_b_character(self, state: NovelGenerationState) -> Dict:
        """
        Agent B：角色对话和描写
        专注于符合角色性格的对话、心理活动、动作描写
        """
        logger.info("Agent B（角色）开始工作")

        # 构建提示词
        prompt = ChatPromptTemplate.from_messages([
            ("system", """你是一位专业的小说角色描写专家。你的任务是根据剧情提示和世界观描写，创作符合角色性格的对话、心理活动和动作描写。

            要求：
            1. 严格遵循角色性格设定
            2. 对话要符合角色说话风格
            3. 心理活动要真实细腻
            4. 控制在200-250字
            5. 基于以下世界观描写继续创作
            6. 直接描写对话和动作，严禁使用"他想表达..."、"她感到..."等概括性语言，要通过细节展示

            角色信息：
            {character_context}

            世界观描写：
            {worldview_output}
            """),
            ("user", "剧情提示：{prompt}\n\n请创作角色的对话、心理和动作描写。")
        ])

        # 调用LLM
        step_start = datetime.utcnow()
        chain = prompt | self.llm_simple
        response = await chain.ainvoke({
            "prompt": state["prompt"],
            "worldview_output": state["worldview_output"],
            "character_context": "\n".join(state.get("character_context", ["无相关角色信息"]))
        })
        step_end = datetime.utcnow()

        character_output = response.content
        logger.info(f"Agent B输出：{character_output[:50]}...")

        steps = state.get("workflow_steps", [])
        step = AgentWorkflowStep(
            id="agent_b_character",
            parent_id="agent_a_worldview",
            type="llm",
            agent_name="AgentBCharacter",
            title="角色描写Agent",
            description="基于世界观描写和角色信息生成对话与心理描写。",
            input={
                "prompt": state["prompt"],
                "worldview_preview": state.get("worldview_output", "")[:80],
            },
            output={
                "preview": character_output[:80],
                "length": len(character_output),
            },
            data_sources={
                "character_context": state.get("character_context", [])[:5],
            },
            llm={
                "model": settings.OPENAI_MODEL_SIMPLE,
                "temperature": 0.7,
            },
            status="completed",
            started_at=step_start,
            finished_at=step_end,
            duration_ms=int((step_end - step_start).total_seconds() * 1000),
        )
        steps.append(step.model_dump())

        return {"character_output": character_output, "workflow_steps": steps}

    async def _agent_c_plot(self, state: NovelGenerationState) -> Dict:
        """
        Agent C：剧情控制
        整合世界观和角色内容，推进剧情，埋设伏笔
        """
        logger.info("Agent C（剧情控制）开始工作")

        # 检查是否有一致性违规需要修正
        consistency_result = state.get("consistency_result", {})
        has_conflict = consistency_result.get("has_conflict", False)
        retry_count = state.get("retry_count", 0)
        
        # 构建基础系统提示词
        system_prompt = """你是一位专业的小说剧情控制专家。你的任务是整合世界观描写和角色内容，推进剧情发展，并适当埋设伏笔。

            要求：
            1. 自然融合世界观描写和角色内容
            2. 推进剧情，不要拖沓
            3. 适当埋设伏笔（如果合适）
            4. 控制在总计{target_length}字左右
            5. 使用自然的段落分行，适当换行，便于阅读，不要刻意把所有内容挤在一整段里
            6. 严禁对剧情进行总结、概述或评价，必须直接描写具体的场景、动作和对话
            7. 不要出现"总而言之"、"综上所述"、"这一章讲述了"等总结性词语
            8. 结尾不要强行升华或总结，保持剧情的自然流动，留有悬念

            世界观描写：
            {worldview_output}

            角色描写：
            {character_output}"""

        # 如果是重试，添加一致性违规信息
        if has_conflict and retry_count > 0:
            violations = consistency_result.get("violations", [])
            if violations:
                violation_text = "\n".join([f"- {v}" for v in violations])
                logger.info(f"Agent C 重试第{retry_count}次，违规信息：{violations}")
                system_prompt += f"""

                    ⚠️ 重要提醒：上一次生成的内容存在一致性问题，请在本次生成中避免以下违规：
                    {violation_text}

                    请特别注意：
                    - 时间线的合理性（角色移动、事件发生的时间间隔）
                    - 地理位置的逻辑性（角色移动距离与时间的匹配）
                    - 角色行为的一致性（不要违反角色设定）
                    - 世界观设定的一致性（不要违反已建立的规则）"""

        # 构建提示词
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("user", "剧情提示：{prompt}\n\n请整合以上内容，输出完整的小说段落。")
        ])

        # 使用复杂模型
        step_start = datetime.utcnow()
        chain = prompt | self.llm_complex
        response = await chain.ainvoke({
            "prompt": state["prompt"],
            "worldview_output": state["worldview_output"],
            "character_output": state["character_output"],
            "target_length": state["target_length"]
        })
        step_end = datetime.utcnow()

        plot_output = response.content
        logger.info(f"Agent C输出：{plot_output[:50]}...（共{len(plot_output)}字）")

        steps = state.get("workflow_steps", [])
        step = AgentWorkflowStep(
            id="agent_c_plot",
            parent_id="agent_b_character",
            type="llm",
            agent_name="AgentCPlot",
            title="剧情控制Agent",
            description="整合世界观与角色内容，生成最终剧情输出。",
            input={
                "prompt": state["prompt"],
                "target_length": state["target_length"],
            },
            output={
                "preview": plot_output[:80],
                "length": len(plot_output),
            },
            data_sources={},
            llm={
                "model": settings.OPENAI_MODEL_COMPLEX,
                "temperature": 0.8,
            },
            status="completed",
            started_at=step_start,
            finished_at=step_end,
            duration_ms=int((step_end - step_start).total_seconds() * 1000),
        )
        steps.append(step.model_dump())

        return {"plot_output": plot_output, "workflow_steps": steps}

    async def _consistency_check(self, state: NovelGenerationState) -> Dict:
        """
        一致性检查节点
        验证生成的内容是否符合世界观规则、角色性格等
        """
        logger.info("执行一致性检查")

        step_start = datetime.utcnow()

        # 调用一致性检查服务
        result = await consistency_service.check_content(
            novel_id=state["novel_id"],
            content=state["plot_output"],
            chapter=state["chapter"],
            current_day=state["current_day"]
        )
        step_end = datetime.utcnow()

        steps = state.get("workflow_steps", [])
        consistency_step = AgentWorkflowStep(
            id="consistency_check",
            parent_id="agent_c_plot",
            type="consistency",
            agent_name="ConsistencyService",
            title="一致性检查",
            description="调用一致性检查服务验证生成内容是否违反世界观或角色设定。",
            input={
                "novel_id": state["novel_id"],
                "chapter": state["chapter"],
                "current_day": state["current_day"],
            },
            output={
                "has_conflict": result.get("has_conflict", False),
                "violation_count": len(result.get("violations", [])),
            },
            data_sources={
                "consistency_layer_results": result.get("layer_results", {}),
                "consistency_workflow_trace": result.get("workflow_trace"),
            },
            llm={},
            status="completed",
            started_at=step_start,
            finished_at=step_end,
            duration_ms=int((step_end - step_start).total_seconds() * 1000),
        )
        steps.append(consistency_step.model_dump())

        return {"consistency_result": result, "workflow_steps": steps}

    def _should_retry(self, state: NovelGenerationState) -> str:
        """
        判断是否需要重试

        Returns:
            "retry" 或 "end"
        """
        result = state.get("consistency_result", {})
        has_conflict = result.get("has_conflict", False)
        retry_count = state.get("retry_count", 0)

        # 如果有冲突且重试次数小于2次，则重试
        # 注意：最多重试2次（总共3次生成），防止无限重试
        if has_conflict and retry_count < 2:
            logger.warning(f"检测到一致性冲突，执行第{retry_count + 1}次重试")
            state["retry_count"] = retry_count + 1
            return "retry"

        if has_conflict:
            logger.warning(f"重试次数已达上限（共{retry_count + 1}次生成），仍存在一致性冲突，将返回最后生成的内容")
            # 不再重试，返回最后生成的内容
            logger.info(f"最后一次生成的内容长度：{len(state.get('plot_output', ''))}字")

        return "end"

    async def generate_content(
        self,
        request: GenerationRequest
    ) -> GenerationResponse:
        """
        生成小说内容

        Args:
            request: 生成请求

        Returns:
            生成响应
        """
        logger.info(f"开始生成内容：小说{request.novel_id}，章节{request.chapter}")

        # 准备初始状态
        initial_state: NovelGenerationState = {
            "novel_id": request.novel_id,
            "prompt": request.prompt,
            "chapter": request.chapter,
            "current_day": request.current_day,
            "target_length": request.target_length,
            "worldview_output": "",
            "character_output": "",
            "plot_output": "",
            "worldview_context": [],
            "character_context": [],
            "consistency_result": {},
            "retry_count": 0,
            "workflow_steps": [],
        }

        # 执行工作流
        final_state = await self.workflow.ainvoke(initial_state)

        # 从一致性结果中构建结构化的一致性检查列表
        consistency_result = final_state.get("consistency_result", {}) or {}
        layer_results = consistency_result.get("layer_results", {}) or {}

        consistency_checks: List[ConsistencyCheckResult] = []

        # 规则引擎
        rule_layer = layer_results.get("rule_engine") or {}
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.RULE_ENGINE,
                is_valid=bool(rule_layer.get("is_valid", True)),
                violations=list(rule_layer.get("violations", [])),
            )
        )

        # 知识图谱
        kg_layer = layer_results.get("knowledge_graph") or {}
        kg_violations = list(kg_layer.get("violations", []))
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.KNOWLEDGE_GRAPH,
                is_valid=not bool(kg_violations),
                violations=kg_violations,
            )
        )

        # 时间线
        timeline_layer = layer_results.get("timeline") or {}
        timeline_is_valid = bool(timeline_layer.get("is_valid", True))
        timeline_violations: List[str] = []
        if not timeline_is_valid and timeline_layer.get("reason"):
            timeline_violations = [str(timeline_layer.get("reason"))]
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.TIMELINE,
                is_valid=timeline_is_valid,
                violations=timeline_violations,
            )
        )

        # 情绪状态机（目前为占位）
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.EMOTION,
                is_valid=True,
                violations=[],
            )
        )

        # 构建Agent工作流追踪
        steps_data = final_state.get("workflow_steps", []) or []
        steps: List[AgentWorkflowStep] = []
        for item in steps_data:
            try:
                steps.append(AgentWorkflowStep(**item))
            except Exception:
                # 忽略单个步骤解析错误，避免影响整体响应
                logger.warning("解析AgentWorkflowStep失败，已跳过一条步骤数据")
                continue

        workflow_trace = AgentWorkflowTrace(
            run_id=f"agent-generate-{request.novel_id}-{request.chapter}-{int(datetime.utcnow().timestamp() * 1000)}",
            trigger="generation.generate_content",
            novel_id=request.novel_id,
            chapter_id=request.chapter,
            user_id=None,
            summary=f"小说{request.novel_id} 第{request.chapter}章的多Agent内容生成",
            steps=steps,
        )

        # 构建响应
        response = GenerationResponse(
            novel_id=request.novel_id,
            chapter=request.chapter,
            final_content=final_state["plot_output"],
            agent_outputs=[
                AgentOutput(
                    agent_type=AgentType.WORLDVIEW,
                    content=final_state["worldview_output"],
                ),
                AgentOutput(
                    agent_type=AgentType.CHARACTER,
                    content=final_state["character_output"],
                ),
                AgentOutput(
                    agent_type=AgentType.PLOT,
                    content=final_state["plot_output"],
                ),
            ],
            consistency_checks=consistency_checks,
            retry_count=final_state["retry_count"],
            generated_at=datetime.now(),
            worldview_context=final_state.get("worldview_context", []),
            character_context=final_state.get("character_context", []),
            workflow_trace=workflow_trace,
        )

        logger.info(f"内容生成完成，共{len(response.final_content)}字")
        return response

    async def generate_content_stream(
        self,
        request: GenerationRequest
    ):
        """
        流式生成小说内容，yield事件
        """
        logger.info(f"开始流式生成内容：小说{request.novel_id}，章节{request.chapter}")

        # 准备初始状态
        initial_state: NovelGenerationState = {
            "novel_id": request.novel_id,
            "prompt": request.prompt,
            "chapter": request.chapter,
            "current_day": request.current_day,
            "target_length": request.target_length,
            "worldview_output": "",
            "character_output": "",
            "plot_output": "",
            "worldview_context": [],
            "character_context": [],
            "consistency_result": {},
            "retry_count": 0,
            "workflow_steps": [],
        }

        # 记录合并后的状态
        final_state = initial_state.copy()

        # yield initial event
        yield {"type": "agent", "agent": "System", "status": "初始化完成", "data": None}

        async for output in self.workflow.astream(initial_state):
            for node_name, node_data in output.items():
                # 更新最终状态
                final_state.update(node_data)
                
                # 根据节点名称发送事件
                if node_name == "retrieve_context":
                    yield {"type": "agent", "agent": "RAG", "status": "上下文检索完成", "data": {"worldview_chunks": len(node_data.get("worldview_context", [])), "character_chunks": len(node_data.get("character_context", []))}}
                    yield {"type": "agent", "agent": "Agent A", "status": "正在构思世界观...", "data": None}
                
                elif node_name == "agent_a_worldview":
                    yield {"type": "agent", "agent": "Agent A", "status": "世界观描写完成", "data": {"preview": node_data.get("worldview_output", "")[:50]}}
                    yield {"type": "agent", "agent": "Agent B", "status": "正在刻画角色...", "data": None}
                
                elif node_name == "agent_b_character":
                    yield {"type": "agent", "agent": "Agent B", "status": "角色描写完成", "data": {"preview": node_data.get("character_output", "")[:50]}}
                    yield {"type": "agent", "agent": "Agent C", "status": "正在生成剧情...", "data": None}
                
                elif node_name == "agent_c_plot":
                    yield {"type": "agent", "agent": "Agent C", "status": "剧情生成完成", "data": {"preview": node_data.get("plot_output", "")[:50]}}
                    yield {"type": "agent", "agent": "Consistency", "status": "正在检查一致性...", "data": None}
                
                elif node_name == "consistency_check":
                    result = node_data.get("consistency_result", {})
                    has_conflict = result.get("has_conflict", False)
                    if has_conflict:
                         yield {"type": "agent", "agent": "Consistency", "status": "发现冲突，准备重试", "data": {"violations": result.get("violations", [])}}
                    else:
                         yield {"type": "agent", "agent": "Consistency", "status": "检查通过", "data": None}

        # 从一致性结果中构建结构化的一致性检查列表
        consistency_result = final_state.get("consistency_result", {}) or {}
        layer_results = consistency_result.get("layer_results", {}) or {}

        consistency_checks: List[ConsistencyCheckResult] = []

        # 规则引擎
        rule_layer = layer_results.get("rule_engine") or {}
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.RULE_ENGINE,
                is_valid=bool(rule_layer.get("is_valid", True)),
                violations=list(rule_layer.get("violations", [])),
            )
        )

        # 知识图谱
        kg_layer = layer_results.get("knowledge_graph") or {}
        kg_violations = list(kg_layer.get("violations", []))
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.KNOWLEDGE_GRAPH,
                is_valid=not bool(kg_violations),
                violations=kg_violations,
            )
        )

        # 时间线
        timeline_layer = layer_results.get("timeline") or {}
        timeline_is_valid = bool(timeline_layer.get("is_valid", True))
        timeline_violations: List[str] = []
        if not timeline_is_valid and timeline_layer.get("reason"):
            timeline_violations = [str(timeline_layer.get("reason"))]
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.TIMELINE,
                is_valid=timeline_is_valid,
                violations=timeline_violations,
            )
        )

        # 情绪状态机（目前为占位）
        consistency_checks.append(
            ConsistencyCheckResult(
                check_type=ConsistencyCheckType.EMOTION,
                is_valid=True,
                violations=[],
            )
        )

        # 构建Agent工作流追踪
        steps_data = final_state.get("workflow_steps", []) or []
        steps: List[AgentWorkflowStep] = []
        for item in steps_data:
            try:
                steps.append(AgentWorkflowStep(**item))
            except Exception:
                logger.warning("解析AgentWorkflowStep失败，已跳过一条步骤数据")
                continue

        workflow_trace = AgentWorkflowTrace(
            run_id=f"agent-generate-{request.novel_id}-{request.chapter}-{int(datetime.utcnow().timestamp() * 1000)}",
            trigger="generation.generate_content_stream",
            novel_id=request.novel_id,
            chapter_id=request.chapter,
            user_id=None,
            summary=f"小说{request.novel_id} 第{request.chapter}章的多Agent内容生成",
            steps=steps,
        )

        # 构建响应
        response = GenerationResponse(
            novel_id=request.novel_id,
            chapter=request.chapter,
            final_content=final_state["plot_output"],
            agent_outputs=[
                AgentOutput(
                    agent_type=AgentType.WORLDVIEW,
                    content=final_state["worldview_output"],
                ),
                AgentOutput(
                    agent_type=AgentType.CHARACTER,
                    content=final_state["character_output"],
                ),
                AgentOutput(
                    agent_type=AgentType.PLOT,
                    content=final_state["plot_output"],
                ),
            ],
            consistency_checks=consistency_checks,
            retry_count=final_state["retry_count"],
            generated_at=datetime.now(),
            worldview_context=final_state.get("worldview_context", []),
            character_context=final_state.get("character_context", []),
            workflow_trace=workflow_trace,
        )

        logger.info(f"流式内容生成完成，共{len(response.final_content)}字")
        yield {"type": "final_response", "data": response}
    
    async def generate_character(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI生成角色"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的小说角色设计师。根据以下信息，创建一个丰富立体的角色。

        小说信息：
        - 标题：{novel_title}
        - 类型：{novel_genre}  
        - 世界观：{worldview}
        
        角色要求：
        {character_requirements}
        
        已有角色：{existing_characters}
        
        请生成一个新角色，包含以下信息：
        1. 姓名（确保不与已有角色重复）
        2. 年龄
        3. 性别
        4. 职业
        5. 外貌描述（100-200字）
        6. 性格特征（100-200字）
        7. 背景故事（200-300字）
        8. 技能列表（3-5项）
        9. 角色弧线（发展轨迹）
        10. 重要性级别（main/secondary/minor）
        
        要求角色符合世界观设定，性格鲜明，有发展潜力。
        
        请以JSON格式返回，字段名使用英文：
        {{
            "name": "角色姓名",
            "age": 年龄数字,
            "gender": "性别",
            "occupation": "职业",
            "appearance": "外貌描述",
            "personality": "性格特征", 
            "background": "背景故事",
            "skills": ["技能1", "技能2", "技能3"],
            "character_arc": "角色弧线",
            "importance_level": "重要性级别"
        }}
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.8
            )
            
            chain = prompt | llm
            response = await chain.ainvoke(context)
            
            # 解析JSON响应
            character_data = json.loads(response.content)
            
            logger.info(f"AI生成角色: {character_data.get('name', 'Unknown')}")
            return character_data
            
        except Exception as e:
            logger.error(f"AI生成角色失败: {str(e)}")
            raise
    
    async def analyze_character(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI分析角色"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的角色分析师。请对以下角色进行深度分析。

        角色信息：
        - 姓名：{name}
        - 年龄：{age}
        - 性格：{personality}
        - 背景：{background}
        - 角色弧线：{character_arc}
        
        关系网络：
        {relationships}
        
        出场记录：
        {appearances}
        
        分析类型：{analysis_type}
        
        请从以下维度进行分析：
        
        1. 性格分析
        - 核心性格特征
        - 性格优缺点
        - 性格一致性评估
        - 性格发展潜力
        
        2. 发展分析  
        - 角色弧线完整性
        - 成长轨迹合理性
        - 发展节奏评估
        - 未来发展建议
        
        3. 关系分析
        - 关系网络复杂度
        - 关系动态变化
        - 关系冲突潜力
        - 关系发展建议
        
        4. 一致性检查
        - 行为一致性
        - 对话风格一致性
        - 价值观一致性
        - 不一致之处识别
        
        5. 改进建议
        - 角色深度提升
        - 特色强化建议
        - 情节参与度优化
        - 读者印象增强
        
        请以JSON格式返回分析结果：
        {{
            "personality_analysis": {{
                "core_traits": ["特征1", "特征2"],
                "strengths": ["优点1", "优点2"], 
                "weaknesses": ["缺点1", "缺点2"],
                "consistency_score": 评分(1-10),
                "development_potential": "发展潜力描述"
            }},
            "development_analysis": {{
                "arc_completeness": 评分(1-10),
                "growth_trajectory": "成长轨迹评估",
                "pacing_assessment": "节奏评估",
                "future_suggestions": ["建议1", "建议2"]
            }},
            "relationship_analysis": {{
                "network_complexity": 评分(1-10),
                "dynamic_changes": "关系变化分析",
                "conflict_potential": "冲突潜力评估",
                "development_suggestions": ["建议1", "建议2"]
            }},
            "consistency_check": {{
                "behavior_consistency": 评分(1-10),
                "dialogue_consistency": 评分(1-10),
                "value_consistency": 评分(1-10),
                "inconsistencies": ["不一致1", "不一致2"]
            }},
            "improvement_suggestions": [
                "改进建议1",
                "改进建议2", 
                "改进建议3"
            ],
            "overall_score": 总体评分(1-10),
            "summary": "总体评价摘要"
        }}
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.3
            )
            
            # 格式化关系和出场信息
            relationships_str = "\n".join([
                f"- 与{rel['target']}的{rel['type']}关系（强度：{rel['strength']}/10）"
                for rel in context.get('relationships', [])
            ]) if context.get('relationships') else "暂无关系记录"
            
            appearances_str = "\n".join([
                f"- 第{app['chapter']}章：{app['type']}出场（重要性：{app['importance']}/10）"
                for app in context.get('appearances', [])
            ]) if context.get('appearances') else "暂无出场记录"
            
            analysis_context = {
                **context['character'],
                'relationships': relationships_str,
                'appearances': appearances_str,
                'analysis_type': context['analysis_type']
            }
            
            chain = prompt | llm
            response = await chain.ainvoke(analysis_context)
            
            # 解析JSON响应
            analysis_result = json.loads(response.content)
            analysis_result['analysis_timestamp'] = datetime.utcnow().isoformat()
            
            logger.info(f"AI角色分析完成: {context['character']['name']}")
            return analysis_result
            
        except Exception as e:
            logger.error(f"AI角色分析失败: {str(e)}")
            raise
    
    async def optimize_character(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI优化角色"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的角色优化师。请根据优化目标，为角色提供具体的优化建议。

        当前角色：
        - 姓名：{name}
        - 性格：{personality}
        - 背景：{background}
        - 技能：{skills}
        - 角色弧线：{character_arc}
        
        优化目标：
        {goals}
        
        需要保持的特征：
        {preserve}
        
        请提供以下优化建议：
        
        1. 性格优化
        - 保持核心特征的同时，如何增加层次感
        - 如何平衡优缺点，使角色更真实
        - 性格细节的丰富化建议
        
        2. 背景优化
        - 背景故事的深化方向
        - 如何增加背景与性格的关联性
        - 背景中可挖掘的情节点
        
        3. 技能优化
        - 技能体系的完善
        - 技能与角色定位的匹配度
        - 新技能的添加建议
        
        4. 弧线优化
        - 角色发展轨迹的改进
        - 关键转折点的设计
        - 成长节奏的调整
        
        5. 具体修改建议
        - 哪些内容需要修改
        - 具体的修改方案
        - 修改后的预期效果
        
        请以JSON格式返回优化建议：
        {{
            "personality_optimization": {{
                "layering_suggestions": ["建议1", "建议2"],
                "balance_improvements": ["改进1", "改进2"],
                "detail_enhancements": ["细节1", "细节2"]
            }},
            "background_optimization": {{
                "deepening_directions": ["方向1", "方向2"],
                "personality_connections": ["关联1", "关联2"],
                "plot_potentials": ["情节点1", "情节点2"]
            }},
            "skills_optimization": {{
                "system_improvements": ["改进1", "改进2"],
                "positioning_match": "匹配度评估",
                "new_skills_suggestions": ["新技能1", "新技能2"]
            }},
            "arc_optimization": {{
                "trajectory_improvements": ["改进1", "改进2"],
                "key_turning_points": ["转折点1", "转折点2"],
                "pacing_adjustments": ["调整1", "调整2"]
            }},
            "specific_modifications": {{
                "content_to_modify": ["内容1", "内容2"],
                "modification_plans": ["方案1", "方案2"],
                "expected_effects": ["效果1", "效果2"]
            }},
            "optimization_priority": ["优先级1", "优先级2", "优先级3"],
            "confidence_score": 置信度评分(0.0-1.0),
            "reasoning": "优化理由和逻辑"
        }}
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.5
            )
            
            # 格式化上下文
            optimization_context = {
                **context['character'],
                'goals': "\n".join([f"- {goal}" for goal in context['goals']]),
                'preserve': "\n".join([f"- {trait}" for trait in context['preserve']]),
                'skills': ", ".join(context['character'].get('skills', []))
            }
            
            chain = prompt | llm
            response = await chain.ainvoke(optimization_context)
            
            # 解析JSON响应
            optimization_result = json.loads(response.content)
            
            logger.info(f"AI角色优化完成: {context['character']['name']}")
            return optimization_result
            
        except Exception as e:
            logger.error(f"AI角色优化失败: {str(e)}")
            raise
    
    async def analyze_worldview(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI分析世界观"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的世界观分析师。请对以下世界观设定进行深度分析。

        小说信息：
        - 标题：{novel_title}
        - 类型：{novel_genre}
        - 世界观：{worldview}
        
        现有设定：
        {existing_settings}
        
        请从以下维度进行分析：
        
        1. 一致性分析
        - 设定内部逻辑一致性
        - 不同设定间的协调性
        - 潜在的逻辑冲突
        
        2. 完整性分析
        - 世界观覆盖范围
        - 缺失的重要设定
        - 需要补充的细节
        
        3. 复杂度分析
        - 设定复杂程度评估
        - 读者理解难度
        - 创作实施难度
        
        4. 独特性分析
        - 创新点识别
        - 与同类作品对比
        - 特色优势分析
        
        5. 实用性分析
        - 对情节发展的支撑
        - 对角色塑造的帮助
        - 对冲突设计的价值
        
        请以JSON格式返回分析结果：
        {{
            "consistency_analysis": {{
                "internal_logic_score": 评分(1-10),
                "coordination_score": 评分(1-10),
                "conflicts": ["冲突1", "冲突2"],
                "consistency_suggestions": ["建议1", "建议2"]
            }},
            "completeness_analysis": {{
                "coverage_score": 评分(1-10),
                "missing_elements": ["缺失1", "缺失2"],
                "detail_gaps": ["细节缺口1", "细节缺口2"],
                "completion_suggestions": ["补充建议1", "补充建议2"]
            }},
            "complexity_analysis": {{
                "complexity_level": "简单/中等/复杂",
                "reader_difficulty": 评分(1-10),
                "creation_difficulty": 评分(1-10),
                "complexity_suggestions": ["建议1", "建议2"]
            }},
            "uniqueness_analysis": {{
                "innovation_score": 评分(1-10),
                "unique_elements": ["特色1", "特色2"],
                "competitive_advantages": ["优势1", "优势2"],
                "differentiation_suggestions": ["建议1", "建议2"]
            }},
            "practicality_analysis": {{
                "plot_support_score": 评分(1-10),
                "character_support_score": 评分(1-10),
                "conflict_value_score": 评分(1-10),
                "practical_suggestions": ["建议1", "建议2"]
            }},
            "overall_assessment": {{
                "total_score": 评分(1-10),
                "strengths": ["优势1", "优势2"],
                "weaknesses": ["弱点1", "弱点2"],
                "priority_improvements": ["优先改进1", "优先改进2"]
            }}
        }}
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.3
            )
            
            chain = prompt | llm
            response = await chain.ainvoke(context)
            
            # 解析JSON响应
            analysis_result = json.loads(response.content)
            analysis_result['analysis_timestamp'] = datetime.utcnow().isoformat()
            
            logger.info(f"AI世界观分析完成: {context.get('novel_title', 'Unknown')}")
            return analysis_result
            
        except Exception as e:
            logger.error(f"AI世界观分析失败: {str(e)}")
            raise
    
    async def optimize_worldview(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI优化世界观"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的世界观设计师。请根据分析结果优化世界观设定。

        当前世界观：
        {worldview}
        
        分析结果：
        {analysis_result}
        
        优化目标：
        {optimization_goals}
        
        请提供以下优化建议：
        
        1. 一致性优化
        - 解决逻辑冲突的方案
        - 统一设定标准
        - 建立一致性规则
        
        2. 完整性优化
        - 补充缺失设定
        - 丰富细节描述
        - 扩展覆盖范围
        
        3. 实用性优化
        - 增强情节支撑
        - 优化角色背景
        - 强化冲突基础
        
        4. 独特性优化
        - 突出创新元素
        - 强化特色优势
        - 提升差异化
        
        请以JSON格式返回优化建议：
        {{
            "consistency_optimizations": {{
                "conflict_resolutions": [
                    {{"conflict": "冲突描述", "solution": "解决方案", "implementation": "实施步骤"}}
                ],
                "unification_rules": ["规则1", "规则2"],
                "standard_guidelines": ["指导原则1", "指导原则2"]
            }},
            "completeness_optimizations": {{
                "new_settings": [
                    {{"category": "分类", "name": "名称", "description": "描述", "importance": "重要性"}}
                ],
                "detail_enhancements": [
                    {{"target": "目标设定", "enhancements": ["增强1", "增强2"]}}
                ],
                "expansion_areas": ["扩展领域1", "扩展领域2"]
            }},
            "practicality_optimizations": {{
                "plot_enhancements": ["情节增强1", "情节增强2"],
                "character_background_improvements": ["背景改进1", "背景改进2"],
                "conflict_foundations": ["冲突基础1", "冲突基础2"]
            }},
            "uniqueness_optimizations": {{
                "innovation_highlights": ["创新亮点1", "创新亮点2"],
                "advantage_amplifications": ["优势放大1", "优势放大2"],
                "differentiation_strategies": ["差异化策略1", "差异化策略2"]
            }},
            "implementation_plan": {{
                "phase_1": ["第一阶段任务1", "第一阶段任务2"],
                "phase_2": ["第二阶段任务1", "第二阶段任务2"],
                "phase_3": ["第三阶段任务1", "第三阶段任务2"]
            }},
            "success_metrics": {{
                "consistency_targets": ["一致性目标1", "一致性目标2"],
                "completeness_targets": ["完整性目标1", "完整性目标2"],
                "quality_indicators": ["质量指标1", "质量指标2"]
            }}
        }}
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.5
            )
            
            chain = prompt | llm
            response = await chain.ainvoke(context)
            
            # 解析JSON响应
            optimization_result = json.loads(response.content)
            
            logger.info(f"AI世界观优化完成: {context.get('novel_title', 'Unknown')}")
            return optimization_result
            
        except Exception as e:
            logger.error(f"AI世界观优化失败: {str(e)}")
            raise
    
    async def generate_worldview_setting(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI生成世界观设定"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的世界观设计师。根据要求生成具体的世界观设定。

        小说信息：
        - 标题：{novel_title}
        - 类型：{novel_genre}
        - 现有世界观：{existing_worldview}
        
        设定要求：
        - 分类：{category}
        - 具体需求：{requirements}
        
        现有相关设定：
        {related_settings}
        
        请生成一个详细的世界观设定，包含：
        1. 设定名称
        2. 详细描述
        3. 运作机制
        4. 相关规则
        5. 对故事的影响
        6. 与其他设定的关联
        
        要求：
        - 与现有世界观保持一致
        - 逻辑自洽，合理可信
        - 有助于情节发展
        - 具有独特性和吸引力
        
        请以JSON格式返回：
        {{
            "name": "设定名称",
            "category": "设定分类",
            "description": "详细描述（200-500字）",
            "mechanism": "运作机制说明",
            "rules": ["规则1", "规则2", "规则3"],
            "story_impact": "对故事的影响",
            "related_connections": ["与设定A的关联", "与设定B的关联"],
            "consistency_rules": ["一致性规则1", "一致性规则2"],
            "importance_level": "重要性级别（high/medium/low）",
            "implementation_suggestions": ["实施建议1", "实施建议2"]
        }}
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.7
            )
            
            chain = prompt | llm
            response = await chain.ainvoke(context)
            
            # 解析JSON响应
            setting_data = json.loads(response.content)
            
            logger.info(f"AI生成世界观设定: {setting_data.get('name', 'Unknown')}")
            return setting_data
            
        except Exception as e:
            logger.error(f"AI生成世界观设定失败: {str(e)}")
            raise
    
    async def analyze_plot_structure(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI分析情节结构"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个专业的情节分析师。请分析小说的情节结构。

        小说信息：
        - 标题：{novel_title}
        - 类型：{novel_genre}
        - 章节数：{chapter_count}
        
        情节元素：
        {plot_elements}
        
        章节概要：
        {chapter_summaries}
        
        请从以下维度分析情节结构：
        
        1. 结构完整性
        - 起承转合是否完整
        - 情节发展是否合理
        - 高潮设置是否恰当
        
        2. 节奏控制
        - 情节推进速度
        - 紧张感营造
        - 缓急交替安排
        
        3. 冲突设计
        - 主要冲突识别
        - 次要冲突分析
        - 冲突解决方式
        
        4. 情节连贯性
        - 前后呼应关系
        - 伏笔铺垫效果
        - 逻辑连接强度
        
        请以JSON格式返回分析结果。
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.3
            )
            
            chain = prompt | llm
            response = await chain.ainvoke(context)
            
            analysis_result = json.loads(response.content)
            analysis_result['analysis_timestamp'] = datetime.utcnow().isoformat()
            
            logger.info(f"AI情节结构分析完成: {context.get('novel_title', 'Unknown')}")
            return analysis_result
            
        except Exception as e:
            logger.error(f"AI情节结构分析失败: {str(e)}")
            raise
    
    async def optimize_novel_comprehensive(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """AI全面优化小说"""
        prompt = ChatPromptTemplate.from_template("""
        你是一个资深的小说编辑和创作指导。请对小说进行全面优化。

        小说信息：
        - 标题：{novel_title}
        - 类型：{novel_genre}
        - 当前状态：{current_status}
        
        分析结果：
        {analysis_results}
        
        优化目标：
        {optimization_goals}
        
        请提供系统性的优化方案，包括：
        
        1. 整体结构优化
        2. 角色发展优化
        3. 情节推进优化
        4. 世界观完善
        5. 文风统一
        6. 读者体验提升
        
        请以JSON格式返回详细的优化计划。
        """)
        
        try:
            llm = ChatOpenAI(
                model=settings.OPENAI_MODEL_COMPLEX,
                api_key=settings.OPENAI_API_KEY,
                base_url=settings.OPENAI_API_BASE,
                temperature=0.4
            )
            
            chain = prompt | llm
            response = await chain.ainvoke(context)
            
            optimization_result = json.loads(response.content)
            optimization_result['optimization_timestamp'] = datetime.utcnow().isoformat()
            
            logger.info(f"AI全面优化完成: {context.get('novel_title', 'Unknown')}")
            return optimization_result
            
        except Exception as e:
            logger.error(f"AI全面优化失败: {str(e)}")
            raise


# 创建全局实例
agent_service = AgentService()

In [ ]:
data = {
    "novel_id": 1,
    "prompt": "艾莉觉醒治愈能力的初期，如果她选择了和罗瑟尔一起逃离现实，是否能让两人的羁绊加速？",
    "chapter": 1,
    "current_day": 1,
    "target_length": 300
}
request_instance = GenerationRequest(**data)
response = await agent_service.generate_content(request_instance)

In [ ]:
response.dict()